# VisTAD: Kaggle GPU Experiment Suite & Technical Audit
### Phase-Diverse VLM4TS Teacher Distillation & Loss Function Benchmark

This notebook reproduces the complete experimental suite on **SWaT** (and other benchmarks), evaluating:
1. **The VLM4TS Teacher**: Multi-panel decoupled rendering and uniform phase diversity (eliminating the legacy fixed $t=50$ phase bias and $\pm 3\sigma$ prompt trap).
2. **Loss Function Benchmark**: Systematic comparison of Point-wise BCE, Focal BCE, Continuous Soft-Dice IoU, Temporal Total Variation (TV) regularizer, and Unified Hybrid loss.
3. **Causal TCN vs. Baseline**: Strict temporal causality (zero future lookahead) and morphology-aware pooling (spikes + plateaus).
4. **Continuous Streaming Evaluation**: Stride 10 continuous sliding-window inference across the 179,968-timestamp held-out test split.
5. **Automated Results Export**: Bundles all logs, metrics JSONs, summary CSVs, and markdown reports into a downloadable `.zip` archive.


In [ ]:
# --- Step 1: Workspace Setup & Repository Initialization ---
import os, sys, shutil, subprocess

repo_dir = "/kaggle/working/VisTAD"
target_branch = "test/vlm"

def get_github_token():
    """Retrieve GitHub token from Kaggle Secrets across common variable names."""
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for key in ["GITHUB_TOKEN", "GH_TOKEN", "github_token", "github_pat", "PAT"]:
            try:
                token = secrets.get_secret(key)
                if token and len(token.strip()) > 0:
                    print(f"Successfully loaded Kaggle secret from key: '{key}'")
                    return token.strip()
            except Exception:
                continue
    except Exception as e:
        print(f"Kaggle secrets module unavailable or unattached: {e}")
    return None

gh_token = get_github_token()
auth_url = f"https://{gh_token}@github.com/rhythm-kedia/VisTAD.git" if gh_token else "https://github.com/rhythm-kedia/VisTAD.git"

print(f"Target branch: {target_branch}")
print(f"Repository destination: {repo_dir}")

# Case 1: Notebook is already running inside VisTAD
if os.path.exists("run_teacher_cache.py"):
    print("Already running inside VisTAD root. Synchronizing branch...")
    !git remote set-url origin {auth_url}
    !git fetch origin {target_branch}
    !git checkout {target_branch}
    !git pull origin {target_branch}

# Case 2: Repository already cloned at /kaggle/working/VisTAD
elif os.path.exists(repo_dir) and os.path.exists(os.path.join(repo_dir, "run_teacher_cache.py")):
    print(f"Found existing repository at {repo_dir}. Updating {target_branch}...")
    os.chdir(repo_dir)
    !git remote set-url origin {auth_url}
    !git fetch origin {target_branch}
    !git checkout {target_branch}
    !git pull origin {target_branch}

# Case 3: Offline dataset fallback at /kaggle/input/vistad-code
elif os.path.exists("/kaggle/input/vistad-code"):
    print("Found VisTAD dataset input in /kaggle/input/vistad-code. Copying...")
    if not os.path.exists(repo_dir):
        shutil.copytree("/kaggle/input/vistad-code", repo_dir)
    os.chdir(repo_dir)

# Case 4: Fresh clone from GitHub
else:
    print(f"Cloning VisTAD (branch: {target_branch}) from GitHub...")
    clone_cmd = f"git clone -b {target_branch} {auth_url} {repo_dir}"
    ret = os.system(clone_cmd)
    if ret != 0:
        print(f"Direct branch clone failed (ret={ret}). Falling back to fetch/checkout...")
        os.system(f"git clone {auth_url} {repo_dir}")
        os.chdir(repo_dir)
        os.system(f"git fetch origin {target_branch} && git checkout {target_branch}")
    else:
        os.chdir(repo_dir)

print(f"\nActive working directory: {os.getcwd()}")
print("Current Git state:")
!git status -s
!git branch -v
!git log -1 --oneline

# Install dependencies
!pip install -q pyyaml pandas scikit-learn tabulate matplotlib seaborn transformers qwen_vl_utils accelerate
print("\nDependencies successfully verified.")


In [ ]:
# --- Step 2: GPU Accelerator & Device Profile ---
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA Device:     {torch.cuda.get_device_name(0)}")
    print(f"Device Count:    {torch.cuda.device_count()}")
    print(f"VRAM Available:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No CUDA GPU detected! Ensure GPU T4 x2 or P100 is selected in Kaggle settings.")


In [ ]:
# --- Step 3: Dataset & Cache Auto-Discovery ---
import os, glob, shutil

os.makedirs("data", exist_ok=True)
os.makedirs("cache", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("Scanning /kaggle/input for dataset files and precomputed caches...")
for root, dirs, files in os.walk("/kaggle/input"):
    for ds in ["swat", "smd", "msl", "gecco"]:
        if ds in dirs and not os.path.exists(f"data/{ds}"):
            src = os.path.join(root, ds)
            try:
                os.symlink(src, f"data/{ds}")
            except Exception:
                shutil.copytree(src, f"data/{ds}")
    for f in files:
        if f.endswith(".pt") and ("cache" in f or "teacher" in f):
            src_pt = os.path.join(root, f)
            dst_pt = os.path.join("cache", f)
            if not os.path.exists(dst_pt):
                try:
                    os.symlink(src_pt, dst_pt)
                except Exception:
                    shutil.copy2(src_pt, dst_pt)

print("Active datasets in data/:", [d for d in ["swat", "smd", "msl", "gecco"] if os.path.exists(f"data/{d}")])
print("Active cache files in cache/:", [os.path.basename(p) for p in glob.glob("cache/*.pt")])


## Section 1: Verification of Data Preprocessing & Causal Student Architecture

Verifies that:
1. `TimeSeriesDataset` applies baseline Z-score standardization $(x - \mu)/\sigma$.
2. All 2,900 SWaT training cached windows are included directly in the data loader.
3. `ResidualDilatedBlock` uses causal lookback slicing (zero future lookahead).
4. `score_scalar` employs morphology-aware pooling ($0.4 \max + 0.3 \text{top}_{10} + 0.3 \text{mean}$).


In [ ]:
# --- Step 4: Verify Causal Architecture & Preprocessing ---
import torch
from src.dataset import load_dataset_arrays, TimeSeriesDataset
from src.models import build_student

raw_data, labels = load_dataset_arrays("swat", "swat_attack", "data")
print(f"Loaded SWaT series: {raw_data.shape} (total steps: {len(raw_data):,})")

# Test TimeSeriesDataset standardization
ds = TimeSeriesDataset(data=raw_data[:2000], labels=labels[:2000], window_size=200, stride=100, normalize=True)
sample = ds[0]
x_sample = sample["x"]
print(f"Window x shape: {x_sample.shape} | Mean: {x_sample.mean().item():.3f} | Std: {x_sample.std().item():.3f}")
assert abs(x_sample.mean().item()) < 1.0, "Normalization failure: data not centered near 0!"

# Test Causal Student Forward Pass
model = build_student("tcn", in_channels=6, hidden_dim=128, num_layers=5, causal=True)
dummy_x = torch.randn(4, 6, 200)
out = model(dummy_x)
print(f"Causal TCN output shapes: scalar={out[score_scalar].shape}, dense={out[logits_dense].shape}")
assert out["logits_dense"].shape == (4, 200), "Dense shape mismatch!"
print("Causal student architecture verified successfully.")


## Section 2: VLM4TS Two-Stage Teacher Cache Generation

Generates the phase-diverse teacher cache on SWaT using:
- **Decoupled Multi-Panel Rendering**: Separate continuous analog subplots from discrete actuator states.
- **Phase-Stratified Sampling**: Windows sampled across onset offsets $\delta \in \{20, 50, 100, 160\}$ and full-plateaus (eliminating the legacy $t=50$ bias).
- **VLM Invariant Reasoning**: Diagnostic prompt targeting invariant breaks, sensor flatlines, and drift ramps.


In [ ]:
# --- Step 5: Generate Phase-Diverse VLM4TS Teacher Cache on SWaT ---
import os

cache_out = "cache/swat_swat_attack_vlm4ts_cache.pt"
legacy_cache = "cache/swat_swat_attack_teacher_cache.pt"

# If precomputed cache exists in input or cache, reuse it; otherwise generate
if os.path.exists(cache_out):
    print(f"VLM4TS cache already exists at {cache_out}.")
else:
    print("Generating VLM4TS phase-diverse cache (500 curriculum windows)...")
    !python -m vlm4ts.generate_cache --dataset swat --entity swat_attack --num_windows 500 --output_path {cache_out}

# Benchmark Legacy Teacher vs. VLM4TS Teacher
print("\nBenchmarking Teacher Representations on Ground-Truth SWaT Attacks...")
!python -m vlm4ts.benchmark_teachers --dataset swat --entity swat_attack --legacy_cache {legacy_cache} --vlm4ts_cache {cache_out}


## Section 3: Systematic Loss Function Suite Benchmark on SWaT

Trains the causal Residual-TCN student across all 5 candidate loss formulations (20 epochs each):
1. **BCE**: Baseline point-wise weighted Binary Cross-Entropy
2. **Focal**: Focal BCE down-weighting easy nominal negatives ($\gamma=2.0$)
3. **Dice**: BCE + Continuous Temporal Soft-Dice IoU (segment overlap)
4. **TV**: BCE + Temporal Total Variation regularizer (smoothness / anti-chatter)
5. **Hybrid**: Unified Composite Loss (Focal + Soft-Dice + TV + Pairwise Margin Ranking)


In [ ]:
# --- Step 6: Train Student Models Across Loss Variants ---
import os, glob

loss_variants = [
    ("hybrid", "tcn_loss_hybrid", 20),
    ("focal",  "tcn_loss_focal",  20),
    ("dice",   "tcn_loss_dice",   20),
    ("tv",     "tcn_loss_tv",     20),
    ("bce",    "tcn_loss_bce",    20),
]

trained_ckpts = {}

for ltype, tag, epochs in loss_variants:
    ckpt_name = f"checkpoints/swat_swat_attack_tcn_{tag}_best.pt"
    print(f"\n{=*70}\n>>> Training SWaT Student with Loss Formulation: {ltype.upper()} ({epochs} epochs)\n{=*70}")
    
    !python train.py --config configs/swat.yaml \
                     --dataset swat \
                     --entity swat_attack \
                     --arch tcn \
                     --num_layers 5 \
                     --epochs {epochs} \
                     --batch_size 64 \
                     --loss_type {ltype} \
                     --tag {tag}
                     
    if os.path.exists(ckpt_name):
        trained_ckpts[ltype] = ckpt_name
        print(f"Successfully saved checkpoint: {ckpt_name}")
    else:
        candidates = glob.glob(f"checkpoints/swat_*{tag}*.pt")
        trained_ckpts[ltype] = candidates[0] if candidates else None

print("\nCompleted training for all loss variants:", trained_ckpts)


## Section 4: Master Continuous Streaming Evaluation (Stride 10)

Runs continuous sliding-window inference (`stride = 10`) across all 179,968 timestamps of the held-out test split for each trained student checkpoint:
- Measures Point AUROC, PR-AUC, VUS-ROC, VUS-PR.
- Measures Operational Point F1, Affiliation Precision, Affiliation Recall, and Affiliation F1.
- Measures Mean Temporal IoU.
- Evaluates operating thresholds: Validation $\tau^*$, Dynamic POT, and SLA False Positive Rate ($\text{FPR} \le 0.05$).


In [ ]:
# --- Step 7: Continuous Streaming Evaluation on Held-Out Test Split ---
import os, glob

eval_results = {}

for ltype, ckpt in trained_ckpts.items():
    if not ckpt or not os.path.exists(ckpt):
        print(f"Skipping {ltype}: Checkpoint not found.")
        continue
        
    tag = f"eval_{ltype}"
    print(f"\n{=*70}\n>>> Evaluating Continuous Stream (stride=10): {ltype.upper()}\n{=*70}")
    
    !python evaluate.py --config configs/swat.yaml \
                        --dataset swat \
                        --entity swat_attack \
                        --checkpoint {ckpt} \
                        --arch tcn \
                        --num_layers 5 \
                        --stride 10 \
                        --thresh_mode val \
                        --tag {tag}


## Section 5: Consolidated Experimental Summary & Table Compilation

Parses all evaluation JSONs and generates the master comparison table.


In [ ]:
# --- Step 8: Compile Master Results Table ---
import os, glob, json
import pandas as pd

eval_files = glob.glob("results/swat_swat_attack_tcn_eval_*.json")
if not eval_files:
    eval_files = glob.glob("results/*swat*eval*.json")

rows = []
for p in sorted(eval_files):
    try:
        with open(p) as f:
            d = json.load(f)
        pm = d.get("point_metrics", {})
        aff_val = d.get("affiliation_metrics_val", {})
        aff_fpr = d.get("affiliation_metrics_fpr05", {})
        vus = d.get("vus_metrics", {})
        
        tag = d.get("tag", os.path.basename(p))
        rows.append({
            "Loss Formulation": tag.replace("eval_", "").upper(),
            "AUROC": pm.get("auroc", 0.0),
            "PR-AUC": pm.get("pr_auc", 0.0),
            "VUS-ROC": vus.get("vus_roc", 0.0),
            "Best Point F1": pm.get("best_f1", 0.0),
            "Oper Point F1": pm.get("f1", 0.0),
            "Oper Precision": pm.get("precision", 0.0),
            "Oper Recall": pm.get("recall", 0.0),
            "Raw Aff-F1": aff_val.get("aff_f1", 0.0),
            "SLA Aff-F1 (FPR<=0.05)": aff_fpr.get("aff_f1", 0.0),
            "Mean Temporal IoU": aff_val.get("mean_tiou", 0.0),
            "Val tau*": d.get("tau_star_threshold", 0.0),
            "SLA tau*": d.get("tau_fpr05_threshold", 0.0)
        })
    except Exception as e:
        print(f"Error reading {p}: {e}")

if rows:
    summary_df = pd.DataFrame(rows).sort_values(by="AUROC", ascending=False)
    print("\n" + "="*85)
    print("             MASTER CONTINUOUS STREAMING BENCHMARK RESULTS (SWAT)")
    print("="*85)
    print(summary_df.to_string(index=False))
    print("="*85 + "\n")
    
    summary_df.to_csv("results/swat_loss_benchmark_summary.csv", index=False)
    summary_df.to_markdown("results/swat_loss_benchmark_summary.md", index=False)
    print("Saved summary table to results/swat_loss_benchmark_summary.csv and .md")


## Section 6: Results Packaging & Download Bundle

Compresses all generated artifacts, logs, evaluation JSONs, and summary tables into a single archive: `vistad_kaggle_results_bundle.zip` in `/kaggle/working/`.


In [ ]:
# --- Step 9: Package Complete Results Bundle for Analysis ---
import os, zipfile, glob

bundle_path = "/kaggle/working/vistad_kaggle_results_bundle.zip"
print(f"Packaging all experimental logs and metrics into {bundle_path}...")

files_to_zip = []
for folder in ["results", "checkpoints", "vlm4ts"]:
    for f in glob.glob(f"{folder}/**/*", recursive=True):
        if not os.path.isdir(f) and not f.endswith(".pt"):  # Exclude large weights from zip for quick download
            files_to_zip.append(f)

for f in ["TODO.md", "README.md"]:
    if os.path.exists(f):
        files_to_zip.append(f)

with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_zip:
        zf.write(f)

print(f"Successfully packaged {len(files_to_zip)} files ({os.path.getsize(bundle_path) / 1024:.1f} KB).")
print(f"Download bundle ready: {bundle_path}")
print("You can download this zip directly from Kaggle Output to provide back the complete logs and evaluation data!")
